In [71]:
import tensorflow as tf
#import tensorflow_decision_forests as tfdf
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.svm import SVC 
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, StackingClassifier
#import xgboost as xgb
#from catboost import CatBoostClassifier
#from lightgbm import LGBMClassifier

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler


train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")
print("Full train dataset shape is {}".format(train_df.shape))

Full train dataset shape is (8693, 14)


# Toutes les modifications

In [72]:
def age_group(df):
    age_group  = []
    for i in df["Age"]:
        if i<=4:
            age_group.append("Age_0-4")
        elif (i>4 and i<=12):
            age_group.append("Age_05-12")
        elif (i>12 and i<=18):
            age_group.append("Age_13-18")
        elif (i>18 and i<=25):
            age_group.append("Age_19-25")
        elif (i>25 and i<=32):
            age_group.append("Age_26-32")
        elif (i>32 and i<=50):
            age_group.append("Age_33_50")
        elif (i>50):
            age_group.append("Age_50+")
        else:
            age_group.append(np.nan)
        
    df["Age Group"] = age_group

age_group(train_df)
age_group(test_df)


def passagerid_new_features(df):
    df["Group"] = df["PassengerId"].apply(lambda x: int(x.split("_")[0]))
    df["Member"] = df["PassengerId"].apply(lambda x: int(x.split("_")[1]))

    x = df.groupby("Group")["Member"].count()
    y = set(x[x>1].index)

    df["Travelling_Solo"] = df["Group"].apply(lambda x : x not in y)
    df["Group_size"] = 0

    for i in x.items():
        df.loc[df["Group"]==i[0], "Group_size"] = i[1]
    df["Group_size"] = df["Group_size"].astype(int)

passagerid_new_features(train_df)
passagerid_new_features(test_df)

def cabin_new_feature(df):
    df["Cabin"].fillna("np.nan/np.nan/np.nan", inplace=True)
    
    df["Cabin_Deck"] = df["Cabin"].apply(lambda x: x.split("/")[0])
    df["Cabin_Number"] = df["Cabin"].apply(lambda x: x.split("/")[1])
    df["Cabin_Side"] = df["Cabin"].apply(lambda x: x.split("/")[2])
    
    # Remplacer les valeurs de chaîne 'np.nan' par des valeurs NaN de numpy
    cols = ["Cabin_Deck", "Cabin_Number", "Cabin_Side"]
    df[cols] = df[cols].replace("np.nan", np.nan)
    
    # Remplir les valeurs manquantes dans les nouvelles caractéristiques créées
    df["Cabin_Deck"].fillna(df["Cabin_Deck"].mode()[0], inplace=True)
    df["Cabin_Side"].fillna(df["Cabin_Side"].mode()[0], inplace=True)
    df["Cabin_Number"] = pd.to_numeric(df["Cabin_Number"], errors='coerce')  # Conversion en numérique
    df["Cabin_Number"].fillna(df["Cabin_Number"].median(), inplace=True)

cabin_new_feature(train_df)
cabin_new_feature(test_df)

def cabin_regions(df):
    df["Cabin_Region1"] = (df["Cabin_Number"]<300)
    df["Cabin_Region2"] = (df["Cabin_Number"]>=300) & (df["Cabin_Number"]<600)
    df["Cabin_Region3"] = (df["Cabin_Number"]>=600) & (df["Cabin_Number"]<900)
    df["Cabin_Region4"] = (df["Cabin_Number"]>=900) & (df["Cabin_Number"]<1200)
    df["Cabin_Region5"] = (df["Cabin_Number"]>=1200) & (df["Cabin_Number"]<1500)
    df["Cabin_Region6"] = (df["Cabin_Number"]>=1500)

cabin_regions(train_df)
cabin_regions(test_df)

exp_cols = ["RoomService","FoodCourt","ShoppingMall","Spa","VRDeck"]
def new_exp_features(df):
    df["Total Expenditure"] = df[exp_cols].sum(axis=1)
    df["No Spending"] = (df["Total Expenditure"]==0)

new_exp_features(train_df)
new_exp_features(test_df)

def expenditure_category(df):
    expense_category = []   
    for i in df["Total Expenditure"]:
        if i==0:
            expense_category.append("No Expense")
        elif (i>0 and i<=716):
            expense_category.append("Low Expense")
        elif (i>716 and i<=1441):
            expense_category.append("Medium Expense")
        elif (i>1441):
            expense_category.append("High Expense")
    df["Expenditure Category"] = expense_category

expenditure_category(train_df)
expenditure_category(test_df)

In [73]:
cat_cols = train_df.select_dtypes(include=["object","bool"]).columns.tolist()
cat_cols.remove("Transported")
num_cols = train_df.select_dtypes(include=["int","float"]).columns.tolist()

def fill_missingno(df):
    df[cat_cols] = SimpleImputer(strategy="most_frequent").fit_transform(df[cat_cols])
    df[num_cols] = SimpleImputer(strategy="median").fit_transform(df[num_cols])

fill_missingno(train_df)
fill_missingno(test_df)

In [74]:
pass_df = test_df[["PassengerId"]]
cols = ["PassengerId","Cabin","Name","Cabin_Number"]

train_df.drop(columns =cols, inplace=True)
test_df.drop(columns=cols, inplace=True)


In [75]:
nominal_cat_cols_one_Hot = ["HomePlanet","Destination"]

train_df = pd.get_dummies(train_df, columns= nominal_cat_cols_one_Hot)
test_df = pd.get_dummies(test_df, columns = nominal_cat_cols_one_Hot)

ordinal_cat_cols_Label = ["CryoSleep","VIP","Travelling_Solo","Cabin_Deck","Cabin_Side","Cabin_Region1","Cabin_Region2",
                    "Cabin_Region3","Cabin_Region4","Cabin_Region5","Cabin_Region6","Age Group","No Spending",
                    "Expenditure Category"]

binary_cols = [col for col in train_df.columns if train_df[col].dropna().unique().size == 2]
new_binary_cols = [col for col in binary_cols if col not in ordinal_cat_cols_Label]

ordinal_cat_cols_Label.extend(new_binary_cols)
ordinal_cat_cols_Label.remove("Transported")

train_df[ordinal_cat_cols_Label] = train_df[ordinal_cat_cols_Label].apply(LabelEncoder().fit_transform)
test_df[ordinal_cat_cols_Label] = test_df[ordinal_cat_cols_Label].apply(LabelEncoder().fit_transform)

Y = train_df["Transported"]
X = train_df.drop(columns=["Transported"])

# Pour commencer à travailler

On a donc X et Y parfaitement initialisés, on fait à présent une normalisation des données.
On aura donc le choix entre X_train et X_train1 pour construire notre modèle de prédiction. Le premier n'est pas normalisé, le second si.

In [76]:
X_scaled = StandardScaler().fit_transform(X)
test_df_scaled = StandardScaler().fit_transform(test_df)

X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2,random_state=0)

X_train1, X_test1, Y_train1, Y_test1 = train_test_split(X_scaled,Y,test_size=0.2,random_state=0)